# 🔄 Notebook 17: ETL Pipeline — Excel → JSON → Database

## สิ่งที่จะได้เรียนรู้
- ETL คืออะไร? (Extract, Transform, Load)
- อ่าน Excel file (Extract)
- ทำความสะอาดและแปลงข้อมูล (Transform)
- แปลงเป็น JSON และบันทึกลง Database (Load)
- Validation — ตรวจสอบข้อมูลก่อนบันทึก
- Error handling — จัดการเมื่อเกิดข้อผิดพลาด

---

> 🐳 ต้องรัน `docker compose up -d` ก่อน (ดู Notebook 15)

### ETL คืออะไร?
```
┌──────────┐    ┌──────────────┐    ┌────────────┐
│ EXTRACT  │ →  │  TRANSFORM   │ →  │   LOAD     │
│          │    │              │    │            │
│ อ่าน Excel│    │ ทำความสะอาด  │    │ บันทึก DB    │
│ อ่าน CSV  │    │ แปลง format  │    │ สร้าง JSON. │
│ อ่าน API  │    │ ตรวจสอบข้อมูล. │    │ สร้าง Report│
└──────────┘    └──────────────┘    └────────────┘
```

### ในระบบ ICP Insight จริง:
```
Excel file → Upload → Validate filename → Extract data → 
Validate elements → Convert to JSON → Save as TestOrder → 
Generate Report Snapshot
```

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import re
from datetime import datetime
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

os.makedirs("output", exist_ok=True)
os.makedirs("output/uploads", exist_ok=True)

# เชื่อมต่อ PostgreSQL (ต้อง docker compose up -d ก่อน)
load_dotenv("../.env")
DATABASE_URL = "postgresql://{}:{}@{}:{}/{}".format(
    os.getenv("DB_USER", "icp_user"),
    os.getenv("DB_PASSWORD", "training_password"),
    os.getenv("DB_HOST", "localhost"),
    os.getenv("DB_PORT", "5432"),
    os.getenv("DB_NAME", "icp_training"),
)
print("✅ Ready")

## Step 0: สร้างข้อมูลตัวอย่าง (สมมติเป็น Excel จาก ICP Lab)

In [ ]:
np.random.seed(42)

# สร้างข้อมูล ICP results (เหมือนที่ได้จากเครื่อง ICP)
samples = []
for i in range(1, 6):
    for rep in [1, 2]:  # replicate 1 and 2
        samples.append({
            "Sample ID": f"S{i:02d}",
            "Replicate": rep,
            "Type": "regular" if i <= 4 else "spike",
            "As (188.979)": round(np.random.uniform(0.1, 3.0), 3),
            "Pb (220.353)": round(np.random.uniform(0.5, 8.0), 3),
            "Cd (226.502)": round(np.random.uniform(0.01, 1.5), 3),
            "Hg (194.168)": round(np.random.uniform(0.001, 0.5), 3),
        })

df_raw = pd.DataFrame(samples)

# บันทึกเป็น Excel (สมมติมาจากเครื่อง ICP)
filename = "EU4_L66_07952.xlsx"
df_raw.to_excel(f"output/{filename}", index=False, sheet_name="Results")
print(f"✅ Created sample Excel: {filename}")
df_raw

## Step 1: EXTRACT — อ่านข้อมูลจาก Excel

### 1.1 Validate Filename ก่อน
ในระบบจริง ชื่อไฟล์ต้องเป็นรูปแบบ: `EU4_L66_07952.xlsx`

In [ ]:
def validate_filename(filename):
    """
    ตรวจสอบชื่อไฟล์ว่าถูกต้องตามรูปแบบ
    Expected: EU4_L66_07952.xlsx
    """
    errors = []
    meta = {"laboratory": None, "panel": None, "sample_number": None}
    
    # ตรวจ extension
    ext = os.path.splitext(filename)[1].lower()
    if ext not in (".xlsx", ".xls"):
        errors.append(f"Invalid file type '{ext}' — ต้องเป็น .xlsx หรือ .xls")
    
    # หา Laboratory code (L66, L67, etc.)
    lab_match = re.search(r"(L\d+)", filename, re.IGNORECASE)
    if lab_match:
        meta["laboratory"] = lab_match.group(1).upper()
    else:
        errors.append("ไม่พบ Laboratory code (เช่น L66, L67)")
    
    # หา Panel type (EU4, EU5, EU9)
    fn_upper = filename.upper()
    for panel in ("EU4", "EU5", "EU9"):
        if panel in fn_upper:
            meta["panel"] = panel
            break
    if not meta["panel"]:
        errors.append("ไม่พบ Panel type (ต้องมี EU4, EU5, หรือ EU9)")
    
    # หา Sample number (5 หลัก)
    num_match = re.search(r"(\d{5})", filename)
    if num_match:
        meta["sample_number"] = num_match.group(1)
    
    return {
        "valid": len(errors) == 0,
        "errors": errors,
        "meta": meta
    }

# ทดสอบ
print("=== Test valid filename ===")
result = validate_filename("EU4_L66_07952.xlsx")
print(f"Valid: {result['valid']}")
print(f"Meta: {result['meta']}")

print("\n=== Test invalid filename ===")
result2 = validate_filename("random_data.csv")
print(f"Valid: {result2['valid']}")
print(f"Errors: {result2['errors']}")

In [ ]:
# อ่าน Excel
filename = "EU4_L66_07952.xlsx"

# Step 1.1: Validate filename
fn_result = validate_filename(filename)
if not fn_result["valid"]:
    print(f"❌ Filename validation failed: {fn_result['errors']}")
else:
    print(f"✅ Filename valid: {fn_result['meta']}")
    
    # Step 1.2: Read Excel
    df = pd.read_excel(f"output/{filename}")
    print(f"\n📊 Read {len(df)} rows, {len(df.columns)} columns")
    print(f"Columns: {list(df.columns)}")
    df.head()

## Step 2: TRANSFORM — แปลงและทำความสะอาดข้อมูล

In [ ]:
def extract_element_info(columns):
    """
    แยกชื่อ element จากชื่อคอลัมน์
    Input:  'As (188.979)'
    Output: {'As': {'wavelength': 188.979, 'column': 'As (188.979)'}}
    """
    elements = {}
    pattern = re.compile(r"^([A-Z][a-z]?)\s*\((\d+\.\d+)\)$")
    
    for col in columns:
        match = pattern.match(col)
        if match:
            symbol = match.group(1)
            wavelength = float(match.group(2))
            elements[symbol] = {
                "wavelength": wavelength,
                "column": col
            }
    
    return elements

element_info = extract_element_info(df.columns)
print("Detected elements:")
for symbol, info in element_info.items():
    print(f"  {symbol}: wavelength={info['wavelength']}, column='{info['column']}'")

In [ ]:
def validate_elements(element_names, expected_panel):
    """
    ตรวจสอบว่า elements ตรงกับ panel ที่ระบุ
    """
    errors = []
    
    # Panel EU4 = 4 elements, EU9 = 9 elements
    expected_count = {"EU4": 4, "EU5": 4, "EU9": 9}.get(expected_panel, 0)
    actual_count = len(element_names)
    
    if expected_count and actual_count != expected_count:
        errors.append(
            f"Expected {expected_count} elements for {expected_panel}, "
            f"but found {actual_count}: {element_names}"
        )
    
    # EU4 ควรมี: As, Pb, Cd, Hg
    if expected_panel in ("EU4", "EU5"):
        required = {"As", "Pb", "Cd", "Hg"}
        missing = required - set(element_names)
        if missing:
            errors.append(f"Missing required elements: {missing}")
    
    return errors

# ตรวจสอบ
elem_errors = validate_elements(list(element_info.keys()), fn_result["meta"]["panel"])
if elem_errors:
    print(f"❌ Element validation failed: {elem_errors}")
else:
    print(f"✅ Elements valid: {list(element_info.keys())} for {fn_result['meta']['panel']}")

In [ ]:
def transform_to_structured(df, element_info, meta):
    """
    แปลง raw DataFrame เป็นโครงสร้าง JSON ที่พร้อมบันทึก
    (แบบเดียวกับที่ระบบ ICP Insight ใช้)
    """
    samples = []
    
    for _, row in df.iterrows():
        # สร้าง concentrations dict
        concentrations = {}
        for symbol, info in element_info.items():
            value = row[info["column"]]
            # จัดการค่า ND (Not Detected)
            if pd.isna(value) or str(value).strip().upper() in ("ND", "N/D", "<LOD"):
                concentrations[symbol] = None
            else:
                concentrations[symbol] = float(value)
        
        sample = {
            "sample_id": str(row["Sample ID"]),
            "replicate": int(row["Replicate"]),
            "type": str(row.get("Type", "regular")),
            "concentrations": concentrations,
        }
        samples.append(sample)
    
    # สร้างโครงสร้างผลลัพธ์
    result = {
        "file_name": meta.get("filename", ""),
        "laboratory": meta["laboratory"],
        "panel": meta["panel"],
        "sample_number": meta["sample_number"],
        "elements": list(element_info.keys()),
        "element_columns": {s: info["column"] for s, info in element_info.items()},
        "samples": samples,
        "metadata": {
            "total_samples": len(samples),
            "processed_at": datetime.now().isoformat(),
            "regular_count": len([s for s in samples if s["type"] == "regular"]),
            "spike_count": len([s for s in samples if s["type"] == "spike"]),
        }
    }
    
    return result

# Transform!
meta = fn_result["meta"]
meta["filename"] = filename

structured_data = transform_to_structured(df, element_info, meta)

print("✅ Transformed data:")
print(f"   Elements: {structured_data['elements']}")
print(f"   Samples: {structured_data['metadata']['total_samples']}")
print(f"   Regular: {structured_data['metadata']['regular_count']}")
print(f"   Spike: {structured_data['metadata']['spike_count']}")

# ดูตัวอย่าง 1 sample
print("\nSample data:")
print(json.dumps(structured_data["samples"][0], indent=2))

In [ ]:
# บันทึก JSON (สำหรับตรวจสอบ)
with open("output/extracted_data.json", "w", encoding="utf-8") as f:
    json.dump(structured_data, f, indent=2, ensure_ascii=False)

print("✅ Saved JSON to output/extracted_data.json")

## Step 3: LOAD — บันทึกลง Database

In [ ]:
# สร้าง Database Engine + Tables
engine = create_engine(DATABASE_URL, echo=False)

with engine.connect() as conn:
    # Drop เพื่อเริ่มใหม่ (สำหรับฝึกเท่านั้น)
    conn.execute(text("DROP TABLE IF EXISTS etl_test_results CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS etl_test_orders CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS etl_laboratories CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS etl_elements CASCADE"))
    
    conn.execute(text("""
        CREATE TABLE etl_elements (
            id SERIAL PRIMARY KEY,
            symbol TEXT NOT NULL UNIQUE,
            name TEXT NOT NULL,
            wavelength REAL,
            unit TEXT DEFAULT 'mg/kg'
        )
    """))
    
    conn.execute(text("""
        CREATE TABLE etl_laboratories (
            id SERIAL PRIMARY KEY,
            code TEXT NOT NULL UNIQUE,
            name TEXT NOT NULL
        )
    """))
    
    conn.execute(text("""
        CREATE TABLE etl_test_orders (
            id SERIAL PRIMARY KEY,
            order_id TEXT NOT NULL UNIQUE,
            sample_id TEXT,
            laboratory_id INTEGER REFERENCES etl_laboratories(id),
            panel_name TEXT,
            order_date TEXT,
            analyst_name TEXT DEFAULT 'Auto-ETL',
            status TEXT DEFAULT 'pending',
            source TEXT DEFAULT 'etl',
            test_results_json TEXT,
            file_path TEXT,
            created_at TEXT
        )
    """))
    
    conn.execute(text("""
        CREATE TABLE etl_test_results (
            id SERIAL PRIMARY KEY,
            order_id INTEGER REFERENCES etl_test_orders(id),
            element_id INTEGER REFERENCES etl_elements(id),
            concentration REAL,
            replicate INTEGER DEFAULT 1
        )
    """))
    
    # เพิ่ม master data
    for sym, name, wl in [("As", "Arsenic", 188.979), ("Pb", "Lead", 220.353),
                           ("Cd", "Cadmium", 226.502), ("Hg", "Mercury", 194.168)]:
        conn.execute(text("INSERT INTO etl_elements (symbol, name, wavelength) VALUES (:s, :n, :w) ON CONFLICT (symbol) DO NOTHING"),
                     {"s": sym, "n": name, "w": wl})
    
    conn.execute(text("INSERT INTO etl_laboratories (code, name) VALUES (:c, :n) ON CONFLICT (code) DO NOTHING"),
                 {"c": "L66", "n": "Lab 66"})
    conn.execute(text("INSERT INTO etl_laboratories (code, name) VALUES (:c, :n) ON CONFLICT (code) DO NOTHING"),
                 {"c": "L67", "n": "Lab 67"})
    conn.commit()

print("✅ Database and master data ready")

In [ ]:
def generate_order_id(engine):
    """สร้าง Order ID ใหม่ (เหมือนระบบจริง)"""
    year = datetime.now().year
    with engine.connect() as conn:
        result = conn.execute(text(
            "SELECT COUNT(*) FROM etl_test_orders WHERE order_id LIKE :pattern"
        ), {"pattern": f"ORD-{year}-%"})
        count = result.fetchone()[0]
    return f"ORD-{year}-{count + 1:04d}"

def load_to_database(engine, structured_data, filename):
    """
    บันทึกข้อมูลที่แปลงแล้วลง Database
    (Simple version ของ process-and-save ในระบบจริง)
    """
    with engine.connect() as conn:
        try:
            # 1. หา laboratory_id
            lab_row = conn.execute(
                text("SELECT id FROM etl_laboratories WHERE code = :code"),
                {"code": structured_data["laboratory"]}
            ).fetchone()
            
            if not lab_row:
                return {"success": False, "error": f"Lab {structured_data['laboratory']} not found"}
            
            lab_id = lab_row[0]
            
            # 2. สร้าง Order ID
            order_id = generate_order_id(engine)
            
            # 3. บันทึก Test Order (พร้อม JSON)
            conn.execute(text("""
                INSERT INTO etl_test_orders 
                (order_id, sample_id, laboratory_id, panel_name, order_date, 
                 source, test_results_json, file_path, created_at)
                VALUES (:oid, :sid, :lid, :panel, :date, 
                        :source, :json, :file, :created)
            """), {
                "oid": order_id,
                "sid": structured_data["sample_number"] or "",
                "lid": lab_id,
                "panel": structured_data["panel"],
                "date": datetime.now().isoformat(),
                "source": "etl",
                "json": json.dumps(structured_data, default=str),
                "file": filename,
                "created": datetime.now().isoformat(),
            })
            
            # 4. หา order row id
            order_row = conn.execute(
                text("SELECT id FROM etl_test_orders WHERE order_id = :oid"),
                {"oid": order_id}
            ).fetchone()
            db_order_id = order_row[0]
            
            # 5. บันทึก test results (แยกเป็น rows)
            result_count = 0
            for sample in structured_data["samples"]:
                for symbol, conc in sample["concentrations"].items():
                    if conc is not None:
                        # หา element_id
                        elem_row = conn.execute(
                            text("SELECT id FROM etl_elements WHERE symbol = :s"),
                            {"s": symbol}
                        ).fetchone()
                        
                        if elem_row:
                            conn.execute(text("""
                                INSERT INTO etl_test_results 
                                (order_id, element_id, concentration, replicate)
                                VALUES (:oid, :eid, :conc, :rep)
                            """), {
                                "oid": db_order_id,
                                "eid": elem_row[0],
                                "conc": conc,
                                "rep": sample["replicate"],
                            })
                            result_count += 1
            
            conn.commit()
            
            return {
                "success": True,
                "order_id": order_id,
                "db_id": db_order_id,
                "results_saved": result_count,
            }
            
        except Exception as e:
            conn.rollback()
            return {"success": False, "error": str(e)}

# Load!
result = load_to_database(engine, structured_data, filename)

if result["success"]:
    print(f"✅ Saved to database!")
    print(f"   Order ID: {result['order_id']}")
    print(f"   Results saved: {result['results_saved']}")
else:
    print(f"❌ Error: {result['error']}")

## Step 4: ตรวจสอบข้อมูลที่บันทึก

In [ ]:
# ดู orders ที่บันทึก
df_orders = pd.read_sql("SELECT id, order_id, sample_id, panel_name, source, created_at FROM etl_test_orders", engine)
print("Orders in database:")
df_orders

In [ ]:
# ดู results แบบละเอียด
df_results = pd.read_sql("""
    SELECT 
        o.order_id,
        e.symbol,
        r.concentration,
        r.replicate
    FROM etl_test_results r
    INNER JOIN etl_test_orders o ON r.order_id = o.id
    INNER JOIN etl_elements e ON r.element_id = e.id
    ORDER BY o.order_id, e.symbol, r.replicate
""", engine)

print(f"Total results: {len(df_results)}")
df_results.head(10)

In [ ]:
# อ่าน JSON กลับจาก Database
df_json = pd.read_sql("SELECT order_id, test_results_json FROM etl_test_orders LIMIT 1", engine)
stored_json = json.loads(df_json["test_results_json"][0])

print("JSON stored in database:")
print(f"  Elements: {stored_json['elements']}")
print(f"  Total samples: {stored_json['metadata']['total_samples']}")
print(f"  First sample: {stored_json['samples'][0]['sample_id']}, "
      f"As={stored_json['samples'][0]['concentrations']['As']}")

## Step 5: รวมทุกอย่างเป็น ETL Function เดียว

In [ ]:
def run_etl_pipeline(filepath, engine):
    """
    ETL Pipeline ครบวงจร:
    1. Validate filename
    2. Read Excel (Extract)
    3. Transform data
    4. Load to database
    
    Returns dict with success/failure info
    """
    filename = os.path.basename(filepath)
    log = []
    
    # Step 1: Validate filename
    log.append(f"📁 Processing: {filename}")
    fn_result = validate_filename(filename)
    if not fn_result["valid"]:
        log.append(f"❌ Filename validation failed: {fn_result['errors']}")
        return {"success": False, "log": log}
    log.append(f"✅ Filename valid: {fn_result['meta']}")
    
    # Step 2: Extract
    try:
        df = pd.read_excel(filepath)
        log.append(f"📊 Read {len(df)} rows, {len(df.columns)} columns")
    except Exception as e:
        log.append(f"❌ Cannot read Excel: {e}")
        return {"success": False, "log": log}
    
    # Step 3: Transform
    element_info = extract_element_info(df.columns)
    if not element_info:
        log.append("❌ No elements detected in columns")
        return {"success": False, "log": log}
    log.append(f"🔬 Detected elements: {list(element_info.keys())}")
    
    elem_errors = validate_elements(list(element_info.keys()), fn_result["meta"]["panel"])
    if elem_errors:
        log.append(f"❌ Element validation: {elem_errors}")
        return {"success": False, "log": log}
    
    meta = fn_result["meta"]
    meta["filename"] = filename
    structured_data = transform_to_structured(df, element_info, meta)
    log.append(f"✅ Transformed: {structured_data['metadata']['total_samples']} samples")
    
    # Step 4: Load
    result = load_to_database(engine, structured_data, filename)
    if result["success"]:
        log.append(f"✅ Saved: Order {result['order_id']}, {result['results_saved']} results")
    else:
        log.append(f"❌ Load failed: {result['error']}")
    
    return {"success": result.get("success", False), "log": log, **result}

# ทดสอบ ETL Pipeline
print("=" * 60)
print("Running ETL Pipeline")
print("=" * 60)

etl_result = run_etl_pipeline(f"output/{filename}", engine)

for msg in etl_result["log"]:
    print(msg)

print(f"\n{'='*60}")
print(f"Result: {'SUCCESS' if etl_result['success'] else 'FAILED'}")

## 📝 สรุป ETL Pipeline

```
Input: Excel file (EU4_L66_07952.xlsx)
  ↓
1. VALIDATE filename → lab, panel, sample_number
  ↓
2. EXTRACT → pd.read_excel() → DataFrame
  ↓
3. TRANSFORM → extract elements, validate, structure JSON
  ↓
4. LOAD → save to test_orders + test_results tables
  ↓
Output: Data in Database (queryable + reportable)
```

### Key Patterns จากระบบจริง:
- **Filename validation** — ตรวจสอบก่อนอ่านไฟล์
- **Element validation** — elements ต้องตรงกับ panel
- **JSON storage** — เก็บ raw data เป็น JSON ใน DB column
- **Normalized results** — แยกผลลัพธ์เป็น rows ใน test_results table
- **Transaction** — rollback ถ้าเกิด error

### Next: Notebook 18 — Mini Labs (ฝึกปฏิบัติ)